# Мини-запуск MPP: небольшой трансформер + Trainer

Загружаем срез данных, собираем маленькую MaskedPlayerModel и гоняем несколько шагов через HuggingFace Trainer.

In [ ]:
import sys
from pathlib import Path

ROOT = Path(".").resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")  # Apple Silicon
else:
    device = torch.device("cpu")
print("Device:", device)
# AMD GPU: нужен PyTorch с ROCm — https://pytorch.org/get-started/locally/ (выбрать ROCm)

## 1. Данные и словари

In [ ]:
from data.preprocessing import preprocess_raw_csv, build_vocab_mappings

raw_path = ROOT / "dataset" / "data_with_dates.csv"
sample_path = ROOT / "notebooks" / "train_sample_raw.csv"
output_dir = str(ROOT / "notebooks" / "train_sample_processed")

df_raw = pd.read_csv(raw_path)
df_raw.to_csv(sample_path, index=False)
df = preprocess_raw_csv(str(sample_path), output_dir)
vocab = build_vocab_mappings(df, output_dir)

print("Матчей (уникальных match_id):", df["match_id"].nunique())
print("players_vocab_size (pad_idx+1):", vocab["player_pad_token_id"] + 1)
print("Число классов для MPP (реальные игроки):", vocab["player_pad_token_id"] - 1)

## 2. Датасет и коллатор

In [ ]:
from data.dataset import MatchDatasetMPP
from data.collator import DataCollatorMPP
from torch.utils.data import Subset

max_seq_length = 36
ds_full = MatchDatasetMPP(
    df,
    player_name2id=vocab["player_name2id"],
    team_name2id=vocab["team_name2id"],
    max_seq_length=max_seq_length,
    player_pad_token_id=vocab["player_pad_token_id"],
    team_pad_token_id=vocab["team_pad_token_id"],
    position_pad_token_id=25,
)

n = len(ds_full)
n_val = max(1, int(n * 0.05))  # 5% eval, как в risingBALLER
n_train = n - n_val
train_dataset = Subset(ds_full, range(0, n_train))
eval_dataset = Subset(ds_full, range(n_train, n))

collator = DataCollatorMPP(
    player_mask_token_id=vocab["player_mask_token_id"],
    mask_percentage=0.25,
)

print("Train матчей:", n_train, "Eval матчей:", n_val)

## 3. Маленькая модель и TrainingArguments

In [ ]:
from models.pretrain import MaskedPlayerModel
from training.trainer import build_training_args, build_trainer
from training.metrics import compute_metrics_mpp

embed_size = 128
model = MaskedPlayerModel(
    embed_size=embed_size,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=39,
    players_vocab_size=vocab["player_pad_token_id"],
    teams_vocab_size=vocab["team_pad_token_id"],
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
model = model.to(device)

# Параметры как в risingBALLER (train_masked_players_prediction.ipynb)
training_config = {
    "output_dir": str(ROOT / "notebooks" / "mpp_mini_output"),
    "num_train_epochs": 2000,
    "per_device_train_batch_size": 64,
    "per_device_eval_batch_size": 64,
    "learning_rate": 1e-4,
    "weight_decay": 0.0,
    "warmup_ratio": 0.0,
    "lr_scheduler_type": "linear",
    "logging_steps": 100,
    "eval_strategy": "steps",
    "eval_steps": 100,
    "save_strategy": "steps",
    "save_steps": 42750,
    "report_to": "tensorboard",
    "seed": 42,
}

train_args = build_training_args(training_config)
trainer = build_trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics_mpp,
    data_collator=collator,
)

print("Trainer готов. Параметров модели:", sum(p.numel() for p in model.parameters()))

## 4. Запуск обучения

In [ ]:
trainer.train()
# Финальная валидация по eval_steps. Отдельно evaluate() после train() в Jupyter ломает NotebookProgressCallback.

In [ ]:
# Опционально: повторный eval после train. Сначала убираем NotebookProgressCallback (он обнуляется в on_train_end).
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)
trainer.evaluate()